Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 23.8 ms, sys: 5.01 ms, total: 28.8 ms
Wall time: 28.2 ms
CPU times: user 11.2 ms, sys: 2.98 ms, total: 14.1 ms
Wall time: 14.1 ms
CPU times: user 3.08 ms, sys: 2 ms, total: 5.08 ms
Wall time: 5.08 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 150
lr = 5.5e-5
wd = 1e-5
alpha = 0.5
beta= 1
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
shared_dropout = 0
outcome_dropout = 0.1
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 0
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")
print (f" activation = {activation}")

 epochs = 150
 learning rate = 5.5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
 early stop start = 0
 activation = <class 'torch.nn.modules.activation.ReLU'>


In [ ]:
import pandas as pd
import numpy as np
import torch
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3]
wd_grid = [0.0, 1e-6, 1e-5, 1e-4]
alpha_grid = [0.1, 0.5, 1.0]
beta_grid = [0.1, 0.5, 1.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

total_cfg = len(lr_grid) * len(wd_grid) * len(alpha_grid) * len(beta_grid)
print(f"🔄 Bắt đầu grid search: {total_cfg} cấu hình")

# 2. Loop over all (lr, wd, alpha, beta) pairs
for grid_lr, grid_wd, grid_alpha, grid_beta in product(lr_grid, wd_grid, alpha_grid, beta_grid):
    run_rows = []
    
    print("\n" + "=" * 110)
    print(
        f"Cấu hình: lr={grid_lr:.1e}, weight_decay={grid_wd:.1e}, "
        f"alpha={grid_alpha:.1f}, beta={grid_beta:.1f}"
    )
    print("=" * 110)

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            beta=grid_beta,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        dragonnet.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p, _, _ = dragonnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred, _, _ = dragonnet.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "alpha": grid_alpha,
            "beta": grid_beta,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)
        
        print(f"  ✔ Seed {SEED} | Val_AUQC={current_val_auqc:.4f} | Test_AUQC={row['AUQC']:.4f}")

    # Aggregate each hyperparameter pair
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "alpha": grid_alpha,
        "beta": grid_beta,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

print("\n" + "=" * 130)
print(f"{'KẾT QUẢ GRID SEARCH (XẾP HẠNG THEO MEAN TEST AUQC)':^130}")
print("=" * 130)

display_cols = [
    "lr", "weight_decay", "alpha", "beta",
    "mean_Val_AUQC", "mean_AUUC", "mean_AUQC", "mean_Lift", "mean_KRCC", "mean_ATE_Err",
    "std_Val_AUQC", "std_AUUC", "std_AUQC", "std_Lift", "std_KRCC", "std_ATE_Err"
]

print(df_grid[display_cols].to_string(
    index=False,
    formatters={
        "lr": "{:.1e}".format,
        "weight_decay": "{:.1e}".format,
        "alpha": "{:.1f}".format,
        "beta": "{:.1f}".format,
        "mean_Val_AUQC": "{:.4f}".format,
        "mean_AUUC": "{:.4f}".format,
        "mean_AUQC": "{:.4f}".format,
        "mean_Lift": "{:.4f}".format,
        "mean_KRCC": "{:.4f}".format,
        "mean_ATE_Err": "{:.4f}".format,
        "std_Val_AUQC": "{:.4f}".format,
        "std_AUUC": "{:.4f}".format,
        "std_AUQC": "{:.4f}".format,
        "std_Lift": "{:.4f}".format,
        "std_KRCC": "{:.4f}".format,
        "std_ATE_Err": "{:.4f}".format
    }
))

print("-" * 130)
print("Best hyperparameters by mean TEST AUQC:")
print(
    f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
    f"alpha={best_cfg['alpha']:.1f}, beta={best_cfg['beta']:.1f}"
)
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} ± {best_cfg['std_AUQC']:.4f}")
print("=" * 130)

ModuleNotFoundError: No module named 'Tarnet'

In [27]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        alpha=alpha, beta=beta,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    dragonnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini


Epoch 1/150 | Base Loss: 1.6978 | Tarreg Loss: 1.826505 | Total Loss: 3.5243 | Val Loss: 1.6961 | Raw Qini: 0.4813 | EMA Trend: 0.4813 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.6924 | Tarreg Loss: 1.817506 | Total Loss: 3.5099 | Val Loss: 1.6909 | Raw Qini: 0.4651 | EMA Trend: 0.4788 | (patience: 1/20)
Epoch 3/150 | Base Loss: 1.6871 | Tarreg Loss: 1.819270 | Total Loss: 3.5064 | Val Loss: 1.6857 | Raw Qini: 0.4598 | EMA Trend: 0.4760 | (patience: 2/20)
Epoch 4/150 | Base Loss: 1.6824 | Tarreg Loss: 1.790963 | Total Loss: 3.4734 | Val Loss: 1.6804 | Raw Qini: 0.4501 | EMA Trend: 0.4721 | (patience: 3/20)
Epoch 5/150 | Base Loss: 1.6771 | Tarreg Loss: 1.769987 | Total Loss: 3.4471 | Val Loss: 1.6750 | Raw Qini: 0.4294 | EMA Trend: 0.4657 | (patience: 4/20)
Epoch 6/150 | Base Loss: 1.6715 | Tarreg Loss: 1.773363 | Total Loss: 3.4449 | Val Loss: 1.6692 | Raw Qini: 0.4256 | EMA Trend: 0.4597 | (patience: 5/20)
Epoch 7/150 | Base Loss: 1.6651 | Tarreg Loss: 1.766147 | Total Los

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/Dragonnet/dragonnet.py:326: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.8088 | Tarreg Loss: 1.022138 | Total Loss: 2.8309 | Val Loss: 1.8080 | Raw Qini: 0.6636 | EMA Trend: 0.6636 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.8036 | Tarreg Loss: 0.990928 | Total Loss: 2.7945 | Val Loss: 1.8018 | Raw Qini: 0.6686 | EMA Trend: 0.6644 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.7969 | Tarreg Loss: 0.986700 | Total Loss: 2.7836 | Val Loss: 1.7955 | Raw Qini: 0.6796 | EMA Trend: 0.6667 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 1.7903 | Tarreg Loss: 0.987010 | Total Loss: 2.7773 | Val Loss: 1.7888 | Raw Qini: 0.6956 | EMA Trend: 0.6710 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 1.7836 | Tarreg Loss: 0.968149 | Total Loss: 2.7517 | Val Loss: 1.7815 | Raw Qini: 0.7171 | EMA Trend: 0.6779 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 1.7750 | Tarreg Loss: 0.972293 | Total Loss: 2.7473 | Val Loss: 1.7734 | Raw Qini: 0.7228 | EMA Trend: 0.6847 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Base Los

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/Dragonnet/dragonnet.py:326: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/Dragonnet/dragonnet.py:326: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.8068 | Tarreg Loss: 3.577002 | Total Loss: 5.3838 | Val Loss: 1.8056 | Raw Qini: 0.3167 | EMA Trend: 0.3167 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.8027 | Tarreg Loss: 3.546067 | Total Loss: 5.3487 | Val Loss: 1.8006 | Raw Qini: 0.3080 | EMA Trend: 0.3154 | (patience: 1/20)
Epoch 3/150 | Base Loss: 1.7970 | Tarreg Loss: 3.535867 | Total Loss: 5.3329 | Val Loss: 1.7954 | Raw Qini: 0.3111 | EMA Trend: 0.3147 | (patience: 2/20)
Epoch 4/150 | Base Loss: 1.7914 | Tarreg Loss: 3.531993 | Total Loss: 5.3234 | Val Loss: 1.7898 | Raw Qini: 0.3130 | EMA Trend: 0.3145 | (patience: 3/20)
Epoch 5/150 | Base Loss: 1.7856 | Tarreg Loss: 3.502552 | Total Loss: 5.2882 | Val Loss: 1.7836 | Raw Qini: 0.3131 | EMA Trend: 0.3143 | (patience: 4/20)
Epoch 6/150 | Base Loss: 1.7793 | Tarreg Loss: 3.488214 | Total Loss: 5.2675 | Val Loss: 1.7768 | Raw Qini: 0.3065 | EMA Trend: 0.3131 | (patience: 5/20)
Epoch 7/150 | Base Loss: 1.7717 | Tarreg Loss: 3.497515 | Total Los

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/Dragonnet/dragonnet.py:326: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.6595 | Tarreg Loss: 0.862003 | Total Loss: 2.5215 | Val Loss: 1.6588 | Raw Qini: 0.5163 | EMA Trend: 0.5163 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.6545 | Tarreg Loss: 0.851138 | Total Loss: 2.5056 | Val Loss: 1.6532 | Raw Qini: 0.5179 | EMA Trend: 0.5165 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.6493 | Tarreg Loss: 0.849385 | Total Loss: 2.4987 | Val Loss: 1.6475 | Raw Qini: 0.5217 | EMA Trend: 0.5173 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 1.6434 | Tarreg Loss: 0.823317 | Total Loss: 2.4667 | Val Loss: 1.6417 | Raw Qini: 0.5179 | EMA Trend: 0.5174 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Base Loss: 1.6376 | Tarreg Loss: 0.822624 | Total Loss: 2.4602 | Val Loss: 1.6354 | Raw Qini: 0.5278 | EMA Trend: 0.5190 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 1.6310 | Tarreg Loss: 0.792330 | Total Loss: 2.4233 | Val Loss: 1.6288 | Raw Qini: 0.5511 | EMA Trend: 0.5238 | ⭐ NEW BEST (peak ≥ trend)
Epoc

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Conversion/Dragonnet/dragonnet.py:326: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
